# Milestone 3 - Task 1: Prepare Features and Model-Ready Dataset

## Objective
This task prepares the Kickstarter dataset for machine learning using findings from Milestone 2.

The goals are to:
- Define the prediction target
- Select launch-time features supported by the EDA
- Remove identifier and potential data-leakage featuers
- Perform simple feature engineering where appropriate 
- Split the data into training and testing sets
- Create a shared preprocessing pipeline for numerical and categorical features

The resulting setup will be reused across the classification models in Milestone 3 so model performance can be compared fairly.

In [16]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("../data/kickstarter_us_filtered.csv")
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (38504, 17)


,project id,name,url,category,subcategory,location,status,goal,pledged,funded percentage,backers,funded date,levels,reward levels,updates,comments,duration
0,39409,WHILE THE TREES SLEEP,http://www.kickstarter.com/projects/emiliesaba...,Film & Video,Short Film,"Columbia, MO",successful,10500.0,11545.0,1.099524,66,"Fri, 19 Aug 2011 19:28:17 -0000",7,"$25,$50,$100,$250,$500,$1,000,$2,500",10,2,30.00
1,126581,Educational Online Trading Card Game,http://www.kickstarter.com/projects/972789543/...,Games,Board & Card Games,"Maplewood, NJ",failed,4000.0,20.0,0.005000,2,"Mon, 02 Aug 2010 03:59:00 -0000",5,"$1,$5,$10,$25,$50",6,0,47.18
2,237090,GETTING OVER - One son's search to finally kno...,http://www.kickstarter.com/projects/charnick/g...,Film & Video,Documentary,"Los Angeles, CA",successful,6000.0,6535.0,1.089167,100,"Sun, 08 Apr 2012 02:14:00 -0000",13,"$1,$10,$25,$30,$50,$75,$85,$100,$110,$250,$500...",4,0,32.22
3,246101,The Launch of FlyeGrlRoyalty &quot;The New Nam...,http://www.kickstarter.com/projects/flyegrlroy...,Fashion,Fashion,"Novi, MI",failed,3500.0,0.0,0.000000,0,"Wed, 01 Jun 2011 15:25:39 -0000",6,"$10,$25,$50,$100,$150,$250",2,0,30.00
4,316217,Dinner Party - a short film about friendship.....,http://www.kickstarter.com/projects/249354515/...,Film & Video,Short Film,"Portland, OR",successful,3500.0,3582.0,1.023331,39,"Wed, 22 Jun 2011 13:33:00 -0000",7,"$5,$25,$50,$100,$250,$500,$1,000",8,0,21.43


In [17]:
# Turn successful (1) and failed (0) to integers for modeling
df["success"] = (df["status"] == "successful").astype(int)
print(df["success"].value_counts())
print("\nSuccess rate:", round(df["success"].mean() * 100, 2), "%")

success
1    21075
0    17429
Name: count, dtype: int64

Success rate: 54.73 %


## Feature Selection Decisions
Based on the Milestone 2 EDA, each column was reviewed to determine whether it should be included in the initial modeling feature set. Features were evaluated based on whether they are available at launch, whether they may introduce target leakage, whether they require feature engineering, and whether the EDA suggested they may contain useful predictive information.

| Feature | Decision | Reason |
|---|---|---|
| `project id` | Exclude | Identifier, not a meaningful campaign characteristic |
| `name` | Exclude initially | Free-text feature would require separate NLP feature engineering |
| `url` | Exclude | Identifier-like field; not a meaningful predictor |
| `category` | Include | Available at launch; EDA showed different success rates |
| `subcategory` | Include | Available at launch; captured additional variation |
| `location` | Feature engineer, then include | Available at launch, but raw city-level location has high cardinality; extract state before modeling |
| `status` | Target | This is the outcome the model is trying to predict |
| `goal` | Include | Available at launch; EDA showed success differences across goal ranges |
| `pledged` | Exclude | Causes target leakage because pledged amount is generated during the campaign and directly reflects progress toward the outcome |
| `funded percentage` | Exclude | Causes target leakage because it directly reflects whether the funding goal was reached |
| `backers` | Exclude | Causes target leakage because the number of backers accumulates after launch |
| `funded date` | Exclude initially | Not prioritized in Milestone 2 and requires additional time-based feature engineering |
| `levels` | Include | Available at launch and associated with success |
| `reward levels` | Exclude initially | Raw string requires additional feature engineering; `levels` already captures tier count |
| `updates` | Exclude | Potential target leakage because updates occur during the campaign |
| `comments` | Exclude | Potential target leakage because comments are generated during the campaign |
| `duration` | Include | Available at launch and associated with success |

In [18]:
# Extract state from location
df["state"] = (
    df["location"]
    .str.rsplit(",", n=1)
    .str[-1]
    .str.strip()
)
print("Unique states:", df["state"].nunique())
df["state"].value_counts().head()

Unique states: 51


state
CA    8220
NY    6639
TX    1847
IL    1720
FL    1410
Name: count, dtype: int64

In [19]:
# Define initial feature set
numerical_features = ["goal", "duration", "levels"]
categorical_features = ["category", "subcategory", "state"]
selected_features = numerical_features + categorical_features

In [20]:
# Information used to make prediction
X = df[selected_features].copy()
# Answer we're trying to predict
y = df["success"].copy()

In [21]:
# Validate the modeling data
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nMissing Values:")
print(X.isnull().sum())

print("\nTarget distribution:")
print(y.value_counts())

Feature matrix shape: (38504, 6)
Target shape: (38504,)

Missing Values:
goal           0
duration       0
levels         0
category       0
subcategory    0
state          0
dtype: int64

Target distribution:
success
1    21075
0    17429
Name: count, dtype: int64


In [22]:
# Display the first few rows of the feature matrix
X.head()

,goal,duration,levels,category,subcategory,state
0,10500.0,30.00,7,Film & Video,Short Film,MO
1,4000.0,47.18,5,Games,Board & Card Games,NJ
2,6000.0,32.22,13,Film & Video,Documentary,CA
3,3500.0,30.00,6,Fashion,Fashion,MI
4,3500.0,21.43,7,Film & Video,Short Film,OR


In [23]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

In [24]:
# Validate the training and testing sets
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining success rate:")
print(y_train.value_counts(normalize=True))

print("\nTesting success rate:")
print(y_test.value_counts(normalize=True))

Training rows: 30803
Testing rows: 7701

Training success rate:
success
1    0.547349
0    0.452651
Name: proportion, dtype: float64

Testing success rate:
success
1    0.547332
0    0.452668
Name: proportion, dtype: float64


In [27]:
# Define the preprocessing steps for numerical and categorical features
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [26]:
# Fit the preprocessor on the training data and transform both training and testing data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (30803, 119)
Processed testing shape: (7701, 119)


## Modeling Preparation Summary
The initial model-ready feature set includes:

### Numerical features
- goal
- duration
- levels

### Categorical features
- category
- subcategory
- state

Campaign state was extracted from the original location field to reduced the high cardinality of city-level locations.

Variables including pledged amount, funded percentage, backers, updates, and comments were excluded because they contain information generated after campaign launch and may introduce data leakage. 

Identifiers fields such as project ID and URL were also excluded. Campaign name and raw reward-level values were not included in the initial feature set because they would require additional NLP or feature-engineering work.

The dataset was split into 80% training and 20% testing data using stratification to preserve the successful/failed campaign distribution. Numerical features are standardized and categorical features are one-hot encoded.

All Milestone 3 models should use this same feature set, train/test split, and preprocessing approach so their performace can be compared fairly. Preprocessing should remain consistent where applicable, with model-specific adjustments made when necessary.